# TD34 vs TD3C — new route, migration and Hormuz gap

| | TD3C | TD34 |
|---|---|---|
| Baltic route | Ras Tanura → Ningbo (MEG–China) | **Gulf of Oman → China** |
| Physical read | Inside / through Hormuz load | **Outside** the strait |
| ICE monthly | `TDL` | `T34` |
| Unit | USD/t | USD/t |

Question: has the book started moving from TD3C to TD34, and what does the basis say about the inside/outside premium?

See `docs/ice_ffa_map.md`. OI only on `FILE_FLAG=Final`.

In [ ]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT.parent / "desk_core"))

from freight.ice import asof, front_n, load_ffa_monthly, total_oi  # noqa: E402

pd.options.display.float_format = lambda x: f"{x:,.2f}"
CACHE = ROOT / "data" / "cache"
REFRESH = True  # T34 is new — pull fresh
START = date(2026, 9, 1)
print("ROOT", ROOT)

ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


In [ ]:
tdl = load_ffa_monthly("TDL", start=START, cache_dir=CACHE, refresh=REFRESH)
t34 = load_ffa_monthly("T34", start=START, cache_dir=CACHE, refresh=REFRESH)

t34_birth = t34["TRADE_DATE"].min()
t34_last = t34["TRADE_DATE"].max()
oi_tdl = tdl.loc[tdl["OI_LOTS"].notna()].copy()
oi_t34 = t34.loc[t34["OI_LOTS"].notna()].copy()

session, snap34 = asof(oi_t34)
_, snap3c = asof(oi_tdl.loc[oi_tdl["TRADE_DATE"] <= session])
# Prefer same calendar day for curve overlay
common = sorted(set(t34["TRADE_DATE"]) & set(tdl["TRADE_DATE"]))
curve_day = common[-1]
front34 = front_n(t34.loc[t34["TRADE_DATE"] == curve_day], 24, curve_day)
front3c = front_n(tdl.loc[tdl["TRADE_DATE"] == curve_day], 24, curve_day)

display(Markdown(
    f"**T34 listed** {t34_birth} → {t34_last} ({t34['TRADE_DATE'].nunique()} sessions) · "
    f"{t34['STRIP'].nunique()} strips · OI session {session} · curve day {curve_day}"
))
pd.DataFrame([
    {"route": "TD3C (TDL)", "sessions": tdl["TRADE_DATE"].nunique(), "strips": tdl["STRIP"].nunique(),
     "last": tdl["TRADE_DATE"].max(), "total_oi_last": oi_tdl.loc[oi_tdl["TRADE_DATE"]==session, "OI_LOTS"].sum()},
    {"route": "TD34 (T34)", "sessions": t34["TRADE_DATE"].nunique(), "strips": t34["STRIP"].nunique(),
     "last": t34["TRADE_DATE"].max(), "total_oi_last": oi_t34.loc[oi_t34["TRADE_DATE"]==session, "OI_LOTS"].sum()},
])

C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


**T34 listed** 2026-10-05 → 2026-10-12 (6 sessions) · 27 strips · OI session 2026-10-12 · curve day 2026-10-12

,route,sessions,strips,last,total_oi_last
0,TD3C (TDL),30,53,2026-10-12,"35,783.00"
1,TD34 (T34),6,27,2026-10-12,"4,737.00"


## Inventory

TD34 is a **monthly-only** Baltic FFA so far (no daily mini / balmo under that name in ICE). Strip list is shorter than TDL (~27 vs ~51 months).

In [ ]:
cmp = front3c.merge(
    front34[["CONTRACT_MONTH", "CONTRACT_CODE", "SETTLE_USD_T", "OI_LOTS", "VOLUME_LOTS"]],
    on="CONTRACT_MONTH",
    how="outer",
    suffixes=("_TD3C", "_TD34"),
).sort_values("CONTRACT_MONTH")
cmp["BASIS_TD3C_MINUS_TD34"] = cmp["SETTLE_USD_T_TD3C"] - cmp["SETTLE_USD_T_TD34"]
display(Markdown(f"### Front curve on **{curve_day}** ($/t)"))
cmp[["CONTRACT_CODE_TD3C", "SETTLE_USD_T_TD3C", "OI_LOTS_TD3C",
     "CONTRACT_CODE_TD34", "SETTLE_USD_T_TD34", "OI_LOTS_TD34",
     "BASIS_TD3C_MINUS_TD34"]].head(12)

### Front curve on **2026-10-12** ($/t)

,CONTRACT_CODE_TD3C,SETTLE_USD_T_TD3C,OI_LOTS_TD3C,CONTRACT_CODE_TD34,SETTLE_USD_T_TD34,OI_LOTS_TD34,BASIS_TD3C_MINUS_TD34
0,V6,278.66,"3,947.00",V6,150.10,15.00,128.56
1,X6,260.71,"3,836.00",X6,134.54,665.00,126.17
2,Z6,238.63,"3,799.00",Z6,126.61,355.00,112.02
3,F7,189.49,"2,993.00",F7,102.12,679.00,87.36
4,G7,185.05,"3,003.00",G7,94.38,679.00,90.67
5,H7,180.44,"3,003.00",H7,86.75,679.00,93.69
6,J7,115.37,"1,060.00",J7,69.75,270.00,45.62
7,K7,115.37,"1,035.00",K7,69.75,270.00,45.62
8,M7,115.37,"1,035.00",M7,69.75,270.00,45.62
9,N7,59.24,630.00,N7,30.50,255.00,28.74


## Migration? Liquidity share

If the book were migrating, we would expect TDL total OI to fall as T34 OI rises, and T34 volume to dominate. Watch both series from T34’s listing date.

In [ ]:
tot3c = total_oi(oi_tdl.loc[oi_tdl["TRADE_DATE"] >= date(2026, 9, 21)])
tot34 = total_oi(oi_t34)

vol3c = (
    tdl.loc[tdl["TRADE_DATE"] >= t34_birth]
    .groupby("TRADE_DATE", as_index=False)["VOLUME_LOTS"].sum()
    .rename(columns={"VOLUME_LOTS": "VOL_TD3C"})
)
vol34 = (
    t34.groupby("TRADE_DATE", as_index=False)["VOLUME_LOTS"].sum()
    .rename(columns={"VOLUME_LOTS": "VOL_TD34"})
)
vol = vol3c.merge(vol34, on="TRADE_DATE", how="outer").sort_values("TRADE_DATE")
vol["SHARE_TD34"] = vol["VOL_TD34"] / (vol["VOL_TD3C"] + vol["VOL_TD34"])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Total OI (lots)", "Daily volume (lots)", "TD34 share of combined volume"),
    horizontal_spacing=0.08,
)
fig.update_layout(
    title=f"Liquidity: TD3C vs TD34 | through {session}",
    template="plotly_white", height=420,
    legend=dict(orientation="h"),
)
fig.add_trace(go.Scatter(x=tot3c["TRADE_DATE"], y=tot3c["OI_LOTS"], name="TDL OI",
                         line=dict(color="#1f4e79", width=2)), 1, 1)
fig.add_trace(go.Scatter(x=tot34["TRADE_DATE"], y=tot34["OI_LOTS"], name="T34 OI",
                         line=dict(color="#c0392b", width=2)), 1, 1)
fig.add_vline(x=str(t34_birth), line_dash="dot", line_color="#7f8c8d", row=1, col=1)

fig.add_trace(go.Bar(x=vol["TRADE_DATE"], y=vol["VOL_TD3C"], name="TDL vol",
                     marker_color="#1f4e79"), 1, 2)
fig.add_trace(go.Bar(x=vol["TRADE_DATE"], y=vol["VOL_TD34"], name="T34 vol",
                     marker_color="#c0392b"), 1, 2)
fig.update_layout(barmode="group")

fig.add_trace(go.Scatter(x=vol["TRADE_DATE"], y=vol["SHARE_TD34"], name="TD34 vol share",
                         mode="lines+markers", line=dict(color="#c0392b")), 1, 3)
fig.update_yaxes(tickformat=".0%", row=1, col=3)
fig.show()

oi0 = tot34.iloc[0]["OI_LOTS"]
oi1 = tot34.iloc[-1]["OI_LOTS"]
tdl0 = tot3c.loc[tot3c["TRADE_DATE"] == t34_birth, "OI_LOTS"]
tdl1 = tot3c.loc[tot3c["TRADE_DATE"] == session, "OI_LOTS"]
tdl0v = float(tdl0.iloc[0]) if len(tdl0) else float("nan")
tdl1v = float(tdl1.iloc[0]) if len(tdl1) else float("nan")
display(Markdown(
    f"T34 OI **{oi0:,.0f} → {oi1:,.0f}** lots (+{oi1-oi0:,.0f}) since listing. "
    f"TDL OI on those days **{tdl0v:,.0f} → {tdl1v:,.0f}** "
    f"(Δ {tdl1v-tdl0v:+,.0f}). "
    f"Peak TD34 volume share **{vol['SHARE_TD34'].max():.0%}** "
    f"(latest {vol['SHARE_TD34'].iloc[-1]:.0%})."
))

T34 OI **605 → 4,737** lots (+4,132) since listing. TDL OI on those days **34,786 → 35,783** (Δ +997). Peak TD34 volume share **88%** (latest 88%).

## Curve and Hormuz gap (TD3C − TD34)

Same delivery month: TD3C loads in the Gulf (Hormuz transit); TD34 loads in the Gulf of Oman (outside). Positive basis = market pricing an **inside premium** (or outside discount) in $/t.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"Curve $/t | {curve_day}", "Front-month basis TD3C − TD34"),
    horizontal_spacing=0.08,
)
fig.update_layout(template="plotly_white", height=420, legend=dict(orientation="h"))

fig.add_trace(go.Scatter(
    x=front3c["CONTRACT_CODE"], y=front3c["SETTLE_USD_T"],
    mode="lines+markers", name="TD3C (TDL)", line=dict(color="#1f4e79", width=2),
), 1, 1)
fig.add_trace(go.Scatter(
    x=front34["CONTRACT_CODE"], y=front34["SETTLE_USD_T"],
    mode="lines+markers", name="TD34 (T34)", line=dict(color="#c0392b", width=2),
), 1, 1)
fig.update_yaxes(title_text="USD/t", row=1, col=1)

# Front-month basis history on common sessions
rows = []
for d in common:
    a = front_n(tdl.loc[tdl["TRADE_DATE"] == d], 1, d)
    b = front_n(t34.loc[t34["TRADE_DATE"] == d], 1, d)
    if a.empty or b.empty:
        continue
    # Align on same contract month when possible
    m = a.iloc[0]["CONTRACT_MONTH"]
    b_same = t34.loc[(t34["TRADE_DATE"] == d) & (t34["CONTRACT_MONTH"] == m)]
    if b_same.empty:
        b_same = b
        m_label = f"{a.iloc[0]['STRIP']}/{b.iloc[0]['STRIP']}"
    else:
        m_label = a.iloc[0]["STRIP"]
    rows.append({
        "TRADE_DATE": d,
        "STRIP": m_label,
        "TD3C": float(a.iloc[0]["SETTLE_USD_T"]),
        "TD34": float(b_same.iloc[0]["SETTLE_USD_T"]),
    })
basis = pd.DataFrame(rows)
basis["GAP"] = basis["TD3C"] - basis["TD34"]

fig.add_trace(go.Scatter(x=basis["TRADE_DATE"], y=basis["TD3C"], name="TD3C front",
                         line=dict(color="#1f4e79")), 1, 2)
fig.add_trace(go.Scatter(x=basis["TRADE_DATE"], y=basis["TD34"], name="TD34 same month",
                         line=dict(color="#c0392b")), 1, 2)
fig.update_yaxes(title_text="USD/t", row=1, col=2)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=basis["TRADE_DATE"], y=basis["GAP"],
    marker_color=["#c0392b" if v > 0 else "#27ae60" for v in basis["GAP"]],
    name="TD3C − TD34",
))
fig2.update_layout(
    title="Inside premium: front-month TD3C minus TD34 ($/t)",
    template="plotly_white", height=320,
    yaxis_title="USD/t",
)
fig2.show()

last = basis.iloc[-1]
display(Markdown(
    f"On **{last['TRADE_DATE']}** ({last['STRIP']}): TD3C **{last['TD3C']:.2f}** vs TD34 **{last['TD34']:.2f}** "
    f"→ gap **{last['GAP']:+.2f} $/t**. "
    f"Mean gap since listing **{basis['GAP'].mean():+.2f}** (min {basis['GAP'].min():+.2f}, max {basis['GAP'].max():+.2f})."
))
basis

On **2026-10-12** (Oct26): TD3C **278.66** vs TD34 **150.10** → gap **+128.56 $/t**. Mean gap since listing **+120.67** (min +90.28, max +128.56).

,TRADE_DATE,STRIP,TD3C,TD34,GAP
0,2026-10-05,Oct26,238.29,148.01,90.28
1,2026-10-06,Oct26,273.82,147.78,126.05
2,2026-10-07,Oct26,278.83,150.90,127.94
3,2026-10-08,Oct26,277.46,152.01,125.45
4,2026-10-09,Oct26,278.48,152.71,125.77
5,2026-10-12,Oct26,278.66,150.10,128.56


## Where the new OI sits (TD34 term structure)

Same pack as the TD3C notebook: latest OI by tenor + day-over-day Δ.

In [ ]:
from freight.ice import oi_delta

prev_dates = sorted(oi_t34["TRADE_DATE"].unique())
prev = prev_dates[-2] if len(prev_dates) >= 2 else session
front = front_n(snap34, 12, session)
prev_snap = oi_t34.loc[oi_t34["TRADE_DATE"] == prev]
delta = oi_delta(prev_snap, front)

fig = go.Figure()
fig.add_trace(go.Bar(x=delta["CONTRACT_CODE"], y=delta["OI_LOTS"], name="OI",
                     marker_color="#1f4e79"))
fig.add_trace(go.Bar(
    x=delta["CONTRACT_CODE"], y=delta["OI_DELTA"].fillna(0), name="Δ",
    marker_color=["#c0392b" if v < 0 else "#27ae60" for v in delta["OI_DELTA"].fillna(0)],
))
fig.update_layout(
    title=f"TD34 OI + Δ | {session} (vs {prev})",
    template="plotly_white", height=360, barmode="overlay",
    yaxis_title="Lots",
)
fig.show()
delta[["CONTRACT_CODE", "STRIP", "SETTLE_USD_T", "OI_LOTS", "OI_DELTA"]]

,CONTRACT_CODE,STRIP,SETTLE_USD_T,OI_LOTS,OI_DELTA
0,V6,Oct26,150.10,15.00,15.00
1,X6,Nov26,134.54,665.00,70.00
2,Z6,Dec26,126.61,355.00,65.00
3,F7,Jan27,102.12,679.00,100.00
4,G7,Feb27,94.38,679.00,100.00
5,H7,Mar27,86.75,679.00,100.00
6,J7,Apr27,69.75,270.00,100.00
7,K7,May27,69.75,270.00,100.00
8,M7,Jun27,69.75,270.00,100.00
9,N7,Jul27,30.50,255.00,100.00


## Read-through

- **Not a full migration yet.** TDL still holds ~35k lots of OI; T34 has grown from hundreds to a few thousand in the first week — meaningful seed, not a replacement.
- **Volume can already rival TD3C on spike days** (e.g. 8 Oct). Watch whether that persists or was listing/roll noise.
- **Basis is the insight for 2026.** TD3C ≫ TD34 in $/t on the front is consistent with an inside-Hormuz / war-risk premium versus GoO load. Use that gap alongside Clarksons inside/outside fixture prints (convert both to $/t or TCE).
- **No DFN-style daily for TD34** in the feed yet — cash proxy stays on Baltic (if licensed) or fixtures until ICE lists a mini.
- Next: join Kpler fixtures GoO vs Ras Tanura liftings to see whether physical barrels are already shifting the way the paper strip suggests.